This is a arithmetic agent. The agent has appropriate mathematical tools which can perform arithmetic operations using BODMAS operations.

The architecture is utiling the created_agent() function so the tool call and tool execution is abstracted.

In [12]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [13]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import AgentMiddleware, HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [14]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from typing import Annotated
from operator import add
from typing import Literal
from langchain.agents.structured_output import ToolStrategy

In [15]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [16]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add] = Field(default_factory=list)
    result_in_figure: float | None = None
    result_with_description: str | None = None

In [17]:
@tool
def add(a: float, b: float) -> float:
  """Add two numbers (a + b)."""
  return a + b


@tool
def subtract(a: float, b: float) -> float:
  """Subtract b from a (a - b)."""
  return a - b


@tool
def multiply(a: float, b: float) -> float:
  """Multiply two numbers (a * b)."""
  return a * b


@tool
def divide(a: float, b: float) -> float:
  """Divide a by b (a / b)."""
  if b == 0:
    return "Error: Division by zero is not allowed."
  return a / b


@tool
def power(base: float, exponent: float) -> float:
  """Calculate base raised to the power of exponent (base ^ exponent)."""
  return base**exponent


@tool
def save_final_result(
    result_in_figure: float,
    result_with_description: str,
    runtime: ToolRuntime,
) -> Command:
  """Save the final calculated value and its user-friendly explanation."""
  return Command(
      update={
          "result_in_figure": result_in_figure,
          "result_with_description": result_with_description,
          "messages": [
              ToolMessage(
                  content="Final result saved.",
                  tool_call_id=runtime.tool_call_id,
              )
          ],
      }
  )


tools = [add, subtract, multiply, divide, power, save_final_result]

In [18]:
class DisableParallelToolCalls(AgentMiddleware):
    def wrap_model_call(self, request, handler):
        return handler(request.override(model_settings={"parallel_tool_calls": False}))

checkpointer = InMemorySaver()

system_prompt = """You are a rigorous BODMAS/PEMDAS mathematical reasoning assistant.
Solve expressions step-by-step using tools.

RULES:
1. Always follow BODMAS order:
   - B: Brackets (Innermost first: () then {} then [])
   - O: Orders / Powers / Exponents and 'of' (treat 'X of Y' as multiply X by Y with high priority)
   - D / M: Division and Multiplication (left-to-right)
   - A / S: Addition and Subtraction (left-to-right)

2. EXECUTE EXACTLY ONE TOOL CALL PER TURN. NEVER attempt multiple tool calls at once.
3. Before calling any tool:
   - State the current expression.
   - Identify the single innermost sub-expression being evaluated.
4. After receiving a tool's output:
   - State the result of that sub-expression.
   - Show the simplified expression with the result substituted in.
5. MANDATORY FINAL STEP: When the expression has been completely evaluated to a single final number, you MUST call 'save_final_result' with:
   - result_in_figure: the final numeric answer
   - result_with_description: a concise explanation of the steps and final value
   Do NOT finish the response without calling save_final_result.
"""

arithmetic_agent = create_agent(
   model=model,
   system_prompt=system_prompt,
   tools=tools,
   state_schema=State,
   middleware=[DisableParallelToolCalls()],
   checkpointer=checkpointer,
)

In [19]:
problem = r"What will be the value of $ 25 - \bigl[20 + \bigl\{12 - (5 + 3 \text{ of } 2) \div 2\bigr\} + 3^{2}\bigr] \times 2 $"

print("[bold cyan]=== Starting Step-by-Step BODMAS Evaluation ===[/bold cyan]\n")

config = {"configurable": {"thread_id": "bodmas-stream-1"}}
step_counter = 1

for chunk in arithmetic_agent.stream(
    {"messages": [HumanMessage(content=problem)]},
    config=config,
    stream_mode="updates",
):
    # Check for model thinking & tool calls
    if "model" in chunk:
        for msg in chunk["model"].get("messages", []):
            if msg.content:
                print(f"[bold yellow]Model Thought:[/bold yellow] {msg.content.strip()}")
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    if tc["name"] != "save_final_result":
                        print(
                            f"[bold green]Step {step_counter}: Calling {tc['name']}[/bold green] with args {tc['args']}"
                        )
                    else:
                        print(
                            f"[bold green]Final Step: Calling {tc['name']}[/bold green] with args {tc['args']}"
                        )

    # Check for tool results
    if "tools" in chunk:
        for msg in chunk["tools"].get("messages", []):
            if msg.name != "save_final_result":
                print(f"[bold magenta]↳ Step {step_counter} Result:[/bold magenta] {msg.content}\n")
                step_counter += 1
            else:
                print(f"[bold cyan]↳ Final Result Saved![/bold cyan]\n")

# Fetch final state snapshot from checkpointer using config
final_state = arithmetic_agent.get_state(config)
print("[bold green]=== Execution Complete ===[/bold green]")
print(f"Result in figure: {final_state.values.get('result_in_figure')}")
print(f"Result with description: {final_state.values.get('result_with_description')}")

=== Starting Step-by-Step BODMAS Evaluation ===

Model Thought: The current expression is: 
[ 25 - \bigl[20 + \bigl\{12 - (5 + 3 \text{ of } 2) \div 2\bigr\} + 3^{2}\bigr] \times 2 \]

The innermost sub-expression to evaluate is: 
[ 3 \text{ of } 2 \]

I will calculate \(3 \times 2\).

Step 1: Calling multiply with args {'a': 3, 'b': 2}

↳ Step 1 Result: 6.0

Model Thought: The result of \(3 \text{ of } 2\) is 6.

The expression now simplifies to:
[ 25 - \bigl[20 + \bigl\{12 - (5 + 6) \div 2\bigr\} + 3^{2}\bigr] \times 2 \]

The next innermost sub-expression to evaluate is:
[ 5 + 6 \]

Step 2: Calling add with args {'a': 5, 'b': 6}

... (24 outputs left out)

Model Thought: The value of the expression \( 25 - \bigl[20 + \bigl\{12 - (5 + 3 \text{ of } 2) \div 2\bigr\} + 
3^{2}\bigr] \times 2 \) is \(-46\).

=== Execution Complete ===

Result in figure: -46.0

Result with description: The expression 25 - [20 + {12 - (5 + 3 of 2) ÷ 2} + 3^2] × 2 simplifies step-by-step to 
-46.

In [20]:
config_invoke = {"configurable": {"thread_id": "bodmas-invoke-1"}}
result = arithmetic_agent.invoke(
    {"messages": [HumanMessage(content=problem)]},
    config=config_invoke
)

print("[bold cyan]=== Step-by-Step Summary from Messages ===[/bold cyan]\n")

# Map tool message responses by tool_call_id
tool_responses = {
    m.tool_call_id: m.content
    for m in result["messages"]
    if isinstance(m, ToolMessage)
}

step = 1
for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            if tc["name"] != "save_final_result":
                tool_res = tool_responses.get(tc.get("id"), "")
                print(f"Step {step}: {tc['name']}({tc['args']}) -> {tool_res}")
                step += 1

print(f"\n[bold green]Final Figure:[/bold green] {result.get('result_in_figure')}")
print(f"[bold green]Final Description:[/bold green] {result.get('result_with_description')}")

=== Step-by-Step Summary from Messages ===

Step 1: multiply({'a': 3, 'b': 2}) -> 6.0

Step 2: add({'a': 5, 'b': 6}) -> 11.0

Step 3: divide({'a': 11, 'b': 2}) -> 5.5

Step 4: subtract({'a': 12, 'b': 5.5}) -> 6.5

Step 5: power({'base': 3, 'exponent': 2}) -> 9.0

... (2 outputs left out)

Step 8: multiply({'a': 35.5, 'b': 2}) -> 71.0

Step 9: subtract({'a': 25, 'b': 71}) -> -46.0

Final Figure: -46.0

Final Description: The expression 25 - [20 + {12 - (5 + 3 of 2) ÷ 2} + 3^2] × 2 evaluates step-by-step to -46.